# Global CDR calculations (Coiled cluster version)
This is a Coiled/Dask-parallel version of `cdr-gcam-globalscaling-constApp.ipynb`. Math and postprocessing are the same -- the only difference is that the sweep is dispatched across a Coiled cluster instead of looping over combos one at a time locally.

This script walks through how we scale up each region's representative run to calculate global CDR outcomes.

**Note:** This same workflow works for the `constApp` cases where cropland is capped at 100 percent and application evolves to track it, and for the cases where it has no cap at all. That's because the cap is imposed by the mass of existing deployments exceeding the mass of GCAM application -- a condition that only emerges in the simulations where we have to increase the rock application to mitigate overshoot of the 100 percent cropland condition. If rock application is truly constant (no overshoot response) then the cap never comes into play.

There are a few different ways we can scale up rock application. All would require the total amount of rock spread in each region to reach whatever GCAM simulated. This script assumes a fixed application rate and solves for the area required to reach the GCAM target.

The representative run's application amount was constructed with the following rules:
1. Assume a fixed application flux for each region (in t/ha/yr) and solve for the area required to match the GCAM amount of rock for that region and year.
2. If the cropland area required exceeds the total area available, increase the application flux accordingly (holding the area constant at 100%).

## Postprocessing in a single region
Since more area can be added to the region's ERW over time, we need to start the new area from "square 1" (t=0). That way ERW that starts in 2040 doesn't implicitly carry pH buffering neutralization that's been ongoing since 2020.

We do this by creating one new "deployment" (or "application cohort") every year (some deployments will spread no rock if total application doesn't increase). Each deployment follows the CDR trajectory of the region's representative run and is assigned an area equal to the change in regional area available for ERW. We ensure that the total amount of rock spread in the region each year aligns with the GCAM requirement.

### Year = 0 calculation
We start with the year 0 condition, letting $j$ refer to year and $i$ to a deployment. Since we only have one deployment in year 0, we spread all of the rock GCAM prescribes to that region and year to a single deployment. Formally,

$$M_\text{D, i} = M_\text{gcam, j}$$

when $i$ and $j$ equal zero. We can then calculate the total area required for the year 0 deployment ($A_\text{D, i}$; ha) because we know the application flux ($R_\text{j}$; t/ha/yr) in year 0 (and every year):

$$A_{D, 0} = \frac{M_\text{D, 0}}{R_0}.$$

Similarly, CDR for year 0 is just the CDR for that single deployment.

### Year > 0 calculation
For each year after year 0, the new annual deployment depends on how much *new* rock is spread relative to what existing deployments are already going to do. For each deployment, $i$, we calculate the mass of rock spread in a year by:

$$M_\text{D, i, j} = R_\text{j} \times A_\text{D, i}$$

Then, we get the amount of rock spread by existing deployments ($\mu_\text{j}$; t/yr) by summing them:

$$\mu_\text{j} = \sum_\text{i=0}^n M_\text{D, i, j}$$

Next, we can calculate how much new rock needs to be spread in year $j$ by comparing what existing deployments will do to what GCAM requires ($M_\text{GCAM, j}$; t/yr):

$$M_\text{new, j} = M_\text{GCAM, j} - \mu_\text{j}.$$

This new rock mass will be spread across the year's new deployment, at an area determined by the deployment's application flux that year. Since new deployments always start at $R_0$, we get:

$$A_\text{D, i} = \frac{M_\text{new, j}}{R_0}.$$

Deployment areas are generally fixed through time, with one exception. Since we rely on a single representative run to simulate many ERW deployments evolving over time, it's possible to reach a condition where $M_\text{new, j}$ is negative. That's because new deployments always start at $R_0$, but the representative rock spreading timeseries was calculating implicitly assuming new area kicks in at $R_j$, which can exceed $R_0$ in regions where there isn't enough cropland to hit $M_\text{GCAM, j}$ if $R_j=R_0$. The lower $R_0$ of the new deployment, compared to $R_j$, leads to a rock deficit that needs to be made up by excess area. When that deployment finally hits $R_j$ later on, it'll have "too much" area and we might see $M_\text{new, j}$ become negative.

There's no clear physical meaning for a negative $M_\text{new, j}$. It's just a scaling quirk that we have to deal with that falls out of our decision to use a single representative run to capture the time-evolution of many emerging "deployments" through time. We expect that quirk to be pretty small, likely much smaller than the uncertainty associated with other model decisions like the specific surface area or CEC. Moreover, as long as GCAM deployments see rock amendments stabilizing for long enough after scale-up (and most do), deployment areas, too, will stabilize back so the total area equals the available cropland area.

Since the more recent deployments are typically responsible for the area (and, later, mass) overshoot, we trim area off those first (first in, first out). If they need to be totally trimmed, then they are "turned off" completely. If $M_\text{new, j}$ becomes positive later, we start with a new deployment rather than turning a trimmed one back on or increasing its area.

We could alternatively explore a last in, first out rule, or a "dormancy" rule where trimmed deployments can come back (picking up where they left off) when $M_\text{new, j}$ turns positive. Generally, we don't expect $M_\text{new, j}$ to become positive again after it first turns negative since all individual deployments see their $R$ staying the same or increasing monotonically.

### Calculating CDR
The area calculations are the tricky part. Once we have those, calculating CDR for a given region and year ($\text{CDR}_\text{j}$; t/yr) is simple. It's just the sum of every deployment's yearly CDR flux (t/ha/yr) multiplied by the deployment's area:

$$\text{CDR}_\text{j} = \sum_{i=0}^n \text{CDR}_\text{rate, i, j} \times A_\text{D, i}$$

### Variables
This postprocessing effort produces or uses the following variables, defined over the following coordinates:
| VAR                   | COORDS                     | DESCRIPTION                    | UNITS          |
|-----------------------|----------------------------|--------------------------------|----------------|
| $R$                   | time, region, deployment   | application flux               | t/ha/yr        |
| $A_\text{D}$          | time, region, deployment   | deployment area                | ha             |
| $A_\text{total, ERW}$ | time, region               | total area for ERW             | ha             |
| $M_\text{GCAM}$       | time, region               | total rock spread              | t              |
| $M_\text{D}$          | time, region, deployment   | rock spread in deployment      | t              |
| $\mu$                 | time, region               | rock spread in old deployments | t              |
| $M_\text{new}$        | time, region               | rock spread in new deployment  | t              |
| $CDR_\text{flux}$     | time, region, deployment   | deployment-specific CDR        | t/ha/yr        |
| $CDR_\text{rate}$     | time, region               | region-specific CDR            | t/yr           |
| $A_\text{frac}$       | time, region               | total ERW area over total cropland area | ha    |


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

import os

# --- read in cdr calculation functions
from ew_workflows import cdr_fxns_postproc as cfp
from ew_workflows.cflx_proc import scepter_rco2
# ---
#
# ----------------------------------------------------------------
# [ FOCUS RUNS ]
# exe_central1p5 ; exe_techBehaviorChange1p5 ; exe_noWeathering1p5
# ----------------------------------------------------------------
#
# ************************************************
# --- [ SCEPTER OUTPUTS ]
EXP_GROUP = "gcam_dgsa_v1.2" # "gcam_opr_full_cec3"
EXP_SET = 'exe_central1p5'
SAVE_GROUP = "gcam"
idx = "001"
DUSTSP = 'gbas'   # just for RCO2 determination
#
# --- read in time-transient and integrated data
datpath = f"s3://carbonplan-carbon-removal/ew-workflows-data/scepter/postprocessed/{SAVE_GROUP}/{EXP_GROUP}/{EXP_SET}_{idx}"
fn_cdr = "ds_cdr.zarr"                  # time-integrated data
fn_cdr_trans = "ds_cdr_transient.zarr"  # time-transient data
# read in
ds = xr.open_zarr(os.path.join(datpath, fn_cdr)).load()
ds_trans = xr.open_zarr(os.path.join(datpath, fn_cdr_trans)).load()

# DGSA runs collapse apprate_base (+ dustrad/cec_factor/qrun_factor) into a single
# "collapsed_inputs" dim, so apprate_base is a (site, collapsed_inputs) data_var
# instead of a coordinate. 
IS_DGSA = 'apprate_base' not in ds.coords
if IS_DGSA:
    ds = ds.set_coords('apprate_base')
    ds_trans = ds_trans.set_coords('apprate_base')

# extract apprates for reading in rock flux data
apprates_arr = np.unique(ds.apprate_base.values)

# ************************************************
# --- [ ROCK FLUX INPUTS ]
RUN_FORMAT = 'one-cell-per-region'
DEG_RES = 1
rock_fn = f'annual_rockflx_densest-ag-region_{DEG_RES}deg.pkl'
ag_fn = f'cropland+gcamRegions_time_{DEG_RES}deg.zarr'

if IS_DGSA:
    # The cropland map are apprate-invariant
    # across the FIXED-apprate pkls, so we reuse the reference pkl/zarr and
    # analytically rescale the apprate-dependent columns.
    REF_APPRATE = 5.0  # matches build_dgsa_climatedirs' default reference apprate
    ref_apprate_name = f'{REF_APPRATE}-tHaYr-appRate-FIXED'
    ref_gcam_dir = f's3://carbonplan-carbon-removal/ew-workflows-data/gcam/apprate_calculated/{RUN_FORMAT}/{EXP_SET}/{ref_apprate_name}'
    df_rock_ref = pd.read_pickle(os.path.join(ref_gcam_dir, rock_fn))
    # drop_encoding
    dsag_ref = xr.open_zarr(os.path.join(ref_gcam_dir, ag_fn)).load().drop_encoding()  # see .load() note above

    dfrock_dict = {}
    dsag_dict = {}
    for apprate in apprates_arr:
        apprate_name = f'{float(apprate)}-tHaYr-appRate-FIXED'
        scale = float(apprate) / REF_APPRATE
        df_this = df_rock_ref.copy()
        # In the FIXED-apprate scenario, actual (rockflx_t_ha) and effective
        # (effective_rate_t_ha) rates are identical by construction, and all
        # other apprate-dependent columns are simple linear/inverse functions
        # of the rate (see calculate-rock-fluxes-from-GCAM.ipynb Scenario 3),
        # so we can rescale everything analytically.
        df_this['effective_rate_t_ha'] = float(apprate)
        df_this['rockflx_t_ha'] = float(apprate)
        df_this['rockflx_Mt_ha'] = df_this['rockflx_t_ha'] / 1e6
        df_this['rockflx_Mt'] = df_rock_ref['rockflx_Mt'] * scale
        df_this['cropland_area_required_ha'] = df_rock_ref['cropland_area_required_ha'] / scale
        df_this['cropland_fraction_required'] = df_rock_ref['cropland_fraction_required'] / scale
        df_this['cropland_fraction_effective'] = df_this['cropland_fraction_required']
        dfrock_dict[apprate_name] = df_this
        dsag_dict[apprate_name] = dsag_ref
else:
    dfrock_dict = {}
    for apprate in apprates_arr:
        apprate_name = f'{apprate}-tHaYr-appRate-FIXED'
        gcam_dir = f's3://carbonplan-carbon-removal/ew-workflows-data/gcam/apprate_calculated/{RUN_FORMAT}/{EXP_SET}/{apprate_name}'
        # [ read in ]
        dfrock_dict[apprate_name] = pd.read_pickle(os.path.join(gcam_dir, rock_fn))

    # ************************************************
    # --- [ CROPLAND MAP ]
    #
    ref_apprate_name = f'{apprates_arr[0]}-tHaYr-appRate-FIXED'
    ref_gcam_dir = f's3://carbonplan-carbon-removal/ew-workflows-data/gcam/apprate_calculated/{RUN_FORMAT}/{EXP_SET}/{ref_apprate_name}'
    dsag_ref = xr.open_zarr(os.path.join(ref_gcam_dir, ag_fn)).load().drop_encoding()  # see .load() note above
    dsag_dict = {f'{apprate}-tHaYr-appRate-FIXED': dsag_ref for apprate in apprates_arr}

## Spin up the Coiled cluster

In [2]:
# --- spin up a Coiled/Dask cluster to parallelize the (site, apprate_base, dustrad, cec_factor)
# sweep -- since every combo is postprocessed independently, we can dispatch them as
# separate tasks across cluster workers instead of looping over them one at a time
# in this notebook's single process.
import coiled
from dask.distributed import Client

N_WORKERS = 30          # bump up/down depending on how many combos you're running
WORKER_VM_TYPES = ["r6i.xlarge"]

cluster = coiled.Cluster(
    name="cdr-gcam-globalscaling-constApp",
    n_workers=N_WORKERS,
    worker_vm_types=WORKER_VM_TYPES,
    scheduler_vm_types=WORKER_VM_TYPES,
    region="us-west-2",
    spot_policy="on-demand",
    tags={"Project": "scepter"},
    # No `container=` here on purpose -- Coiled's package sync mirrors your local
    # Python environment (including the editable `ew_workflows` install), so
    # workers run the same ew_workflows code you're running locally instead of
    # whatever version happens to be baked into a container image.
)
client = Client(cluster)
client

[2026-09-21 17:03:22,579][INFO    ][coiled] Fetching latest package priorities...
[2026-09-21 17:03:22,580][INFO    ][coiled.package_sync] Resolving your local /Users/tylerkukla/Github/iam-rtm-compare/uv.lock Python environment...
[2026-09-21 17:03:22,808][INFO    ][coiled.package_sync] Scanning 239 python packages...
[2026-09-21 17:03:23,025][INFO    ][coiled] Running pip check...
[2026-09-21 17:03:23,269][INFO    ][coiled] Validating environment...
[2026-09-21 17:03:24,917][INFO    ][coiled] Creating wheel for ~/Github/ew-workflows/src...
[2026-09-21 17:03:25,004][INFO    ][coiled] Creating wheel for ew-workflows...
[2026-09-21 17:03:25,913][INFO    ][coiled] Uploading coiled_local_src...
[2026-09-21 17:03:26,994][INFO    ][coiled] Uploading ew-workflows...
[2026-09-21 17:03:27,944][INFO    ][coiled] Creating software environment...
[2026-09-21 17:03:28,615][INFO    ][coiled] Creating Cluster (name: cdr-gcam-globalscaling-constApp, https://cloud.coiled.io/clusters/2083452 ). This usu

<Client: 'tls://10.1.35.247:8786' processes=4 threads=16, memory=118.56 GiB>

## Set up postprocessing decisions

In [3]:
# ---- cdr var used for global calculations (don't worry, you can calc globe for other cdr values after)
cdr_var  = 'cdr_adv'
# ---- skip the initial numerical blip (known SCEPTER quirk) ----
t_start = 0.2        # [yr] blip resolved by t ~= 0.1; we use 0.2 for safety margin
# ---- set up the rco2 correction factor
RCO2_GCAM = 0.3      # [tCDR/tRock] cdr potential under perfect efficiency
RCO2_SCEPTER = scepter_rco2[DUSTSP]  # [tCDR/tRock] cdr potential under perfect efficiency
rco2_correction_factor = RCO2_GCAM / RCO2_SCEPTER
# ---- which data to start with (time-integrated or transient)
USE_INTEGRATED = True   # [] True to derive transient data from time-integrated data (mitigate stroboscoping aliasing from transient sampling in edge cases)

In [4]:
# Variables from ds_rep to carry through to every deployment
# varlist_deploy          : interpolated to the annual grid and stored as (time, deployment) lag matrices
# varlist_region          : subset of varlist_deploy → area-scaled regional total  (nansum(var * A_D))
# varist_region_cumfrac   : subset of varlist_deploy → cumulative rock-weighted regional mean (for feedstock fraction vars)
# varlist_rco2_correction : variables we scale to match GCAM's RCO2

_catbudget_vars = [
    'catbudget_gbas', 'catbudget_adv', 'catbudget_tflx',
    'catbudget_sic', 'catbudget_other', 'catbudget_residual',
]

varlist_deploy      = [    # e.g. ['total_dissolution', 'fraction_total_dissolved']
    'cdr_adv', 'cdr_dif', 'cdr_dif_component', 'cdr_resp_component', 'cdr_resp_component_noAE',
    'cdr_SIConly', 'co2pot_adv_tonHaYr_camg', 'co2pot_adv_tonHaYr_totcat', 'co2pot_tot_tonHaYr_camg',
    'co2pot_tot_tonHaYr_totcat', 'DICpot_adv_camg', 'DICpot_adv_totcat', 'fraction_remaining_dissolved',
    'fraction_sld_advected', 'fraction_sld_remaining', 'fraction_total_dissolved',
    'int_dust_ton_ha_yr', 'noncarbsld_source_ca', 'noncarbsld_source_mg', 'noncarbsld_source_k',
    'noncarbsld_source_na', 'total_dissolution',
] + _catbudget_vars

varlist_region      = [      # e.g. ['total_dissolution']
    'cdr_adv', 'cdr_dif', 'cdr_dif_component', 'cdr_resp_component', 'cdr_resp_component_noAE',
    'cdr_SIConly', 'co2pot_adv_tonHaYr_camg', 'co2pot_adv_tonHaYr_totcat', 'co2pot_tot_tonHaYr_camg',
    'co2pot_tot_tonHaYr_totcat',
] + _catbudget_vars

varist_region_cumfrac = [   # e.g. ['fraction_total_dissolved']
    'fraction_remaining_dissolved',
    'fraction_sld_advected', 'fraction_sld_remaining', 'fraction_total_dissolved',
]
# catbudget_* vars are NOT fractions — do not add here

varlist_rco2_correction = [
    'cdr_adv', 'cdr_dif', 'cdr_dif_component', 'cdr_resp_component', 'cdr_resp_component_noAE',
    'cdr_SIConly', 'co2pot_adv_tonHaYr_camg', 'co2pot_adv_tonHaYr_totcat', 'co2pot_tot_tonHaYr_camg',
    'co2pot_tot_tonHaYr_totcat', 'DICpot_adv_camg', 'DICpot_adv_totcat', 'noncarbsld_source_ca',
    'noncarbsld_source_mg', 'noncarbsld_source_k', 'noncarbsld_source_na',
] + _catbudget_vars
# all catbudget terms are proportional to basalt dissolution potential (0.41 t CO2/t rock)
# and need to be scaled to the target rock potential (0.3 t CO2/t rock)

## Run the postprocess (parallelized across the cluster)

In [ ]:
# --- one (site, apprate_base, dustrad, cec_factor) combo -- mirrors the loop body of
# ew_workflows.cdr_fxns_postproc.gcam_postprocess_all_dims_constApp, but returns after a
# single combo so it can be dispatched as one task per cluster worker.
def _process_one_combo(
    isel_dict, ds_trans, ds, dfrock_dict, dsag,
    loop_dims, site_dim, region_col, apprate_key_fn,
    varlist_deploy, varlist_region, varlist_region_cumfrac,
    varlist_rco2_correction, rco2_correction_factor,
    cdr_var, t_start, use_integrated,
):
    from ew_workflows import cdr_fxns_postproc as cfp

    ds_rep_meta = ds_trans.isel(**isel_dict)
    region_name = str(ds_rep_meta[site_dim].values)

    # Use apprate_base coord value whether it's a dim or was scalar-isel'd out;
    # next(iter(dfrock_dict)) is only a safe fallback when apprate_base doesn't exist at all.
    if 'apprate_base' in ds_rep_meta.coords:
        apprate_key = apprate_key_fn(float(ds_rep_meta['apprate_base'].values))
    else:
        apprate_key = next(iter(dfrock_dict))

    df_this     = dfrock_dict[apprate_key]
    region_rows = df_this[df_this[region_col] == region_name]
    if region_rows.empty:
        return None
    region_idx = int(region_rows['reg_id'].iloc[0])

    # Select and (optionally) differentiate the representative run
    if use_integrated:
        ds_rep = cfp._int_to_flux(ds.isel(**isel_dict))
    else:
        ds_rep = ds_rep_meta

    ds_out = cfp.gcam_postprocess_region_constApp(
        ds_rep                  = ds_rep,
        df_rock                 = dfrock_dict[apprate_key],
        dsag                    = dsag,
        region_idx              = region_idx,
        varlist_deploy          = varlist_deploy,
        varlist_region          = varlist_region,
        varlist_region_cumfrac  = varlist_region_cumfrac,
        varlist_rco2_correction = varlist_rco2_correction,
        rco2_correction_factor  = rco2_correction_factor,
        cdr_var                 = cdr_var,
        t_start                 = t_start,
    )

    for dim in loop_dims:
        ds_out = ds_out.assign_coords({dim: ds_rep_meta[dim].values})
    ds_out = ds_out.expand_dims(loop_dims)

    return ds_out

In [ ]:
# --- dispatch every combo to the cluster, gather, and assemble -- same end result as
# cfp.gcam_postprocess_all_dims_constApp.
import itertools
from distributed import progress

loop_dims  = [d for d in ds_trans.dims if d != 'time']
dim_ranges = [range(ds_trans.sizes[d]) for d in loop_dims]
isel_dicts = [dict(zip(loop_dims, idxs)) for idxs in itertools.product(*dim_ranges)]
print(f'Dispatching {len(isel_dicts)} combos across the cluster...')

# scatter the shared inputs once so they aren't re-serialized/re-embedded with every task.
ds_trans_f    = client.scatter(ds_trans, broadcast=True)
ds_f          = client.scatter(ds, broadcast=True)
dfrock_dict_f = client.scatter([dfrock_dict], broadcast=True)[0]

# dsag is loaded on a worker straight from S3 instead of scattered
def _load_dsag(zarr_path):
    import xarray as xr
    return xr.open_zarr(zarr_path).load().drop_encoding()

dsag_f = client.submit(_load_dsag, os.path.join(ref_gcam_dir, ag_fn), pure=False)

futures = client.map(
    _process_one_combo,
    isel_dicts,
    ds_trans                = ds_trans_f,
    ds                       = ds_f,
    dfrock_dict              = dfrock_dict_f,
    dsag                     = dsag_f,
    loop_dims                = loop_dims,
    site_dim                 = 'site',
    region_col               = 'region_nospaces',
    apprate_key_fn           = lambda v: f'{float(v)}-tHaYr-appRate-FIXED',
    varlist_deploy           = varlist_deploy,
    varlist_region           = varlist_region,
    varlist_region_cumfrac   = varist_region_cumfrac,
    varlist_rco2_correction  = varlist_rco2_correction,
    rco2_correction_factor   = rco2_correction_factor,
    cdr_var                  = cdr_var,
    t_start                  = t_start,
    use_integrated           = USE_INTEGRATED,
)

progress(futures)
results_raw = client.gather(futures)
results = [r for r in results_raw if r is not None]
print(f'\n{len(results)} / {len(isel_dicts)} combos returned data (rest skipped -- not in df_rock)')

Dispatching 1750 combos across the cluster...

1750 / 1750 combos returned data (rest skipped -- not in df_rock)


In [ ]:
# --- assemble the gathered per-combo results into one Dataset
ds_all = xr.combine_by_coords(results, combine_attrs='override')

# same derived globals that gcam_postprocess_all_dims_constApp adds at the end
ds_all = ds_all.assign({
    'gcam_cdr_globe': ds_all['gcam_cdr'].sum(dim='site').assign_attrs(
        units='tCO2/yr', description='Global GCAM CDR (sum across active regions)'
    ),
    f'{cdr_var}_globe': ds_all[f'{cdr_var}_region'].sum(dim='site').assign_attrs(
        units='t/yr', description=f'Global modeled CDR ({cdr_var}), sum across active regions'
    ),
    'M_D_region': ds_all['M_D'].sum(dim='deployment').assign_attrs(
        units='t/yr', description='Total rock spread per region per year (sum of M_D across deployments)'
    ),
})

print('\nActive sites:', list(ds_all.site.values))
print('Output dims:', dict(ds_all.sizes))

ds_all


Active sites: [np.str_('Brazil'), np.str_('China'), np.str_('EU-15'), np.str_('India'), np.str_('South_Asia'), np.str_('Southeast_Asia'), np.str_('USA')]
Output dims: {'site': 7, 'collapsed_inputs': 250, 'time': 80, 'deployment': 80}


<xarray.Dataset> Size: 3GB
Dimensions:                                     (site: 7,
                                                 collapsed_inputs: 250,
                                                 time: 80, deployment: 80)
Coordinates:
  * site                                        (site) <U14 392B 'Brazil' ......
  * collapsed_inputs                            (collapsed_inputs) <U125 125kB ...
  * time                                        (time) float64 640B 0.0 ... 79.0
  * deployment                                  (deployment) int64 640B 0 ... 79
Data variables: (12/65)
    cdr_adv_deployment                          (site, collapsed_inputs, time, deployment) float64 90MB ...
    rockflx_t_ha_yr                             (site, collapsed_inputs, time, deployment) float64 90MB ...
    rco2_correction_factor                      (site, collapsed_inputs) float64 14kB ...
    cdr_dif_deployment                          (site, collapsed_inputs, time, deployment) float64 90MB ...
    cdr_dif_component_deployment                (site, collapsed_inputs, time, deployment) float64 90MB ...
    cdr_resp_component_deployment               (site, collapsed_inputs, time, deployment) float64 90MB ...
    ...                                          ...
    total_dissolution_region                    (site, collapsed_inputs, time) float64 1MB ...
    int_dust_ton_ha_yr_region                   (site, collapsed_inputs, time) float64 1MB ...
    fraction_total_dissolved_fromIntFlx_region  (site, collapsed_inputs, time) float64 1MB ...
    gcam_cdr_globe                              (collapsed_inputs, time) float64 160kB ...
    cdr_adv_globe                               (collapsed_inputs, time) float64 160kB ...
    M_D_region                                  (site, collapsed_inputs, time) float64 1MB ...
Attributes:
    region:   Brazil

In [ ]:
# --- reattach swept DGSA parameters as plain variables (not coordinates)
#
if IS_DGSA:
    sample_combo = str(ds['collapsed_inputs'].values[0])
    param_cols = [kv.split('=')[0].strip() for kv in sample_combo.split('|')]
    param_cols = [c for c in param_cols if c in ds.variables]
    ds_params = ds[param_cols].reset_coords()  # demote any that got promoted to coords (e.g. apprate_base) back to variables
    # ds's variables carry zarr v3 encoding (e.g. encoding['serializer']) inherited
    # from ds_cdr.zarr -- incompatible with the zarr_format=2 save below -- so strip
    # it before merging in.
    ds_params = ds_params.drop_encoding()
    ds_all = ds_all.merge(ds_params, join='left')

# Separate out just the regional / global data
Just to save a smaller version of the file

In [9]:
ds_sum = ds_all.drop_dims("deployment")
ds_sum

<xarray.Dataset> Size: 34MB
Dimensions:                                     (site: 7,
                                                 collapsed_inputs: 250, time: 80)
Coordinates:
  * site                                        (site) <U14 392B 'Brazil' ......
  * collapsed_inputs                            (collapsed_inputs) <U125 125kB ...
  * time                                        (time) float64 640B 0.0 ... 79.0
Data variables: (12/38)
    rco2_correction_factor                      (site, collapsed_inputs) float64 14kB ...
    M_GCAM                                      (site, collapsed_inputs, time) float64 1MB ...
    mu                                          (site, collapsed_inputs, time) float64 1MB ...
    M_new                                       (site, collapsed_inputs, time) float64 1MB ...
    A_total_ERW                                 (site, collapsed_inputs, time) float64 1MB ...
    cropland_ha_total                           (site, collapsed_inputs) float64 14kB ...
    ...                                          ...
    cdr_adv_globe                               (collapsed_inputs, time) float64 160kB ...
    M_D_region                                  (site, collapsed_inputs, time) float64 1MB ...
    dustrad                                     (site, collapsed_inputs) float64 14kB ...
    cec_factor                                  (site, collapsed_inputs) float64 14kB ...
    apprate_base                                (site, collapsed_inputs) float64 14kB ...
    qrun_factor                                 (site, collapsed_inputs) float64 14kB ...
Attributes:
    region:   Brazil

## Extra step: application -> dissolution -> CDR lag (cation-budget basis)

Optional diagnostic: for every combination of the swept dims, computes how many years elapse between rock **application**, rock **dissolution**, and **realized CDR** (cation-budget basis), per region and globally.

- `lag_ds` -- per-region lag, dims `(site, ...swept dims...)`.
- `global_lag_ds` -- global lag, dims `(metric, ...swept dims...)`, with `'global_curve_summed'` / `'cdr_weighted_regional_avg'` as variables.

Runs on `ds_sum`, not `ds_all` -- the lag calculation only needs region-level variables (`catbudget_*_region`, `M_GCAM`), not the `(time, deployment)` arrays.

In [10]:
# --- [ EXTRA STEP: application -> dissolution -> CDR lag, cation-budget basis ]
RCO2 = RCO2_GCAM   # tCO2/tRock -- same target stoichiometric potential used above
PERCENTILES = [0.25, 0.50, 0.75]

# mirrors the loop body of ew_workflows.cdr_fxns_postproc.compute_lag_table_all_dims,
# but returns after a single combo so it can be dispatched as one task per cluster worker.
def _process_one_lag_combo(sel_kwargs, dsall, RCO2, percentiles, site_dim):
    from ew_workflows import cdr_fxns_postproc as cfp
    import xarray as xr

    lag_df, region_curves = cfp.compute_lag_table(
        dsall, RCO2=RCO2, percentiles=percentiles, site_dim=site_dim, **sel_kwargs
    )
    global_lag_df = cfp.compute_global_lag(region_curves, lag_df, percentiles=percentiles)

    lag_piece    = xr.Dataset.from_dataframe(lag_df)
    global_piece = xr.Dataset.from_dataframe(global_lag_df)
    loop_dims = list(sel_kwargs.keys())
    for dim, val in sel_kwargs.items():
        lag_piece    = lag_piece.assign_coords({dim: val})
        global_piece = global_piece.assign_coords({dim: val})
    lag_piece    = lag_piece.expand_dims(loop_dims)
    global_piece = global_piece.expand_dims(loop_dims)
    return lag_piece, global_piece


loop_dims_lag = [d for d in ds_sum.dims if d not in ('site', 'time', 'deployment')]
combos = list(itertools.product(*[ds_sum[d].values for d in loop_dims_lag]))
sel_kwargs_list = [dict(zip(loop_dims_lag, combo)) for combo in combos]
print(f'Dispatching {len(sel_kwargs_list)} lag combos across the cluster...')

ds_sum_f = client.scatter(ds_sum, broadcast=True)

lag_futures = client.map(
    _process_one_lag_combo,
    sel_kwargs_list,
    dsall       = ds_sum_f,
    RCO2        = RCO2,
    percentiles = PERCENTILES,
    site_dim    = 'site',
)
progress(lag_futures)
lag_results = client.gather(lag_futures)

lag_pieces    = [r[0] for r in lag_results]
global_pieces = [r[1] for r in lag_results]
lag_ds        = xr.combine_by_coords(lag_pieces, combine_attrs='override')
global_lag_ds = xr.combine_by_coords(global_pieces, combine_attrs='override')
ds_sum = ds_sum.merge(lag_ds).merge(global_lag_ds)

print('lag_ds dims:', dict(lag_ds.sizes))
print('global_lag_ds dims:', dict(global_lag_ds.sizes))
ds_sum

Dispatching 250 lag combos across the cluster...
lag_ds dims: {'collapsed_inputs': 250, 'site': 7}
global_lag_ds dims: {'collapsed_inputs': 250, 'metric': 15}


<xarray.Dataset> Size: 34MB
Dimensions:                                     (site: 7,
                                                 collapsed_inputs: 250,
                                                 time: 80, metric: 15)
Coordinates:
  * site                                        (site) <U14 392B 'Brazil' ......
  * collapsed_inputs                            (collapsed_inputs) <U125 125kB ...
  * time                                        (time) float64 640B 0.0 ... 79.0
  * metric                                      (metric) object 120B 't_app_c...
Data variables: (12/55)
    rco2_correction_factor                      (site, collapsed_inputs) float64 14kB ...
    M_GCAM                                      (site, collapsed_inputs, time) float64 1MB ...
    mu                                          (site, collapsed_inputs, time) float64 1MB ...
    M_new                                       (site, collapsed_inputs, time) float64 1MB ...
    A_total_ERW                                 (site, collapsed_inputs, time) float64 1MB ...
    cropland_ha_total                           (site, collapsed_inputs) float64 14kB ...
    ...                                          ...
    t_diss_cdr_adv_region_P75                   (collapsed_inputs, site) float64 14kB ...
    t_app_diss_region_P75                       (collapsed_inputs, site) float64 14kB ...
    t_app_cdr_adv+sic_region_P75                (collapsed_inputs, site) float64 14kB ...
    t_diss_cdr_adv+sic_region_P75               (collapsed_inputs, site) float64 14kB ...
    global_curve_summed                         (collapsed_inputs, metric) float64 30kB ...
    cdr_weighted_regional_avg                   (collapsed_inputs, metric) float64 30kB ...
Attributes:
    region:   Brazil

## Save the result
Save to the same postproc folder that we read the data in from.

In [11]:
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++
# --- save result
SAVE_FN_ALL = "ds_cdr_globalScaling_constApp_ALL.zarr"
SAVE_FN_SUM = "ds_cdr_globalScaling_constApp_SUM.zarr"
ds_all.to_zarr(os.path.join(datpath, SAVE_FN_ALL), zarr_format=2)
ds_sum.to_zarr(os.path.join(datpath, SAVE_FN_SUM), zarr_format=2)
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++

In [12]:
# --- tear down the cluster now that we're done with it
client.close()
cluster.close()

[2026-09-21 17:11:40,553][INFO    ][coiled] Cluster 2083452 deleted successfully.


In [ ]:
# ---